# Chapter 7: Modeling Sequences With RNNs And Transformers
Language arrives in order, but meaning does not always sit at the surface. A word can lean on what came before it, depend on what comes later, or change force because of the clause wrapped around it. To model language, time series, or any ordered stream, a neural network must learn not just what is present, but how context is carried, revised, and related across a sequence. Recurrent networks and transformers offer two very different answers to that problem.

# Listing 7-1: A simple LSTM model that trains on historical NVDA prices and plots the network’s forecast against the actual market data
This example uses a stock price series as a familiar time series to demonstrate how an LSTM learns temporal patterns. The goal is to capture trends, not to predict individual market fluctuations. This is not a financial forecasting recommendation.)

In [ ]:
# ----------------------------------------------------------------------
# Step 1: Setup and Imports
# ----------------------------------------------------------------------
!pip install yfinance matplotlib scikit-learn torch --quiet
import yfinance as yf
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Step 2: Download and scale the time-series data
# ----------------------------------------------------------------------
def get_timeseries(ticker, start, end, seq_len=20):
    data = yf.download(ticker, start=start, end=end)
    prices = data[['Close']].values.astype('float32')
    scaler = MinMaxScaler()
    # -----------------------------------------------------------
    # Step 2a. In strict forecasting pipelines, scalers are fit
    #          on training data only
    # -----------------------------------------------------------
    scaled = scaler.fit_transform(prices)

    X, y = [], []
    for i in range(len(scaled) - seq_len):
        X.append(scaled[i:i+seq_len])
        y.append(scaled[i+seq_len])
    return torch.tensor(X), torch.tensor(y), scaler, data.index[seq_len:]

# ----------------------------------------------------------------------
# Step 3: Define the LSTM model
# ----------------------------------------------------------------------
class PriceLSTM(nn.Module):
    def __init__(self, hidden_size=50):
        super().__init__()
        # input_size=1 (one feature: closing price)
        # hidden_size controls the dimensionality of the internal state
        # batch_first=True means input shape is (batch, sequence_length, features)
        self.lstm = nn.LSTM(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        # out has shape (batch_size, sequence_length, hidden_size)
        # out[:, -1, :] selects the final time step for each sequence
        return self.fc(out[:, -1, :])

# ----------------------------------------------------------------------
# Step 4: Train the model
# ----------------------------------------------------------------------
def train_model(ticker='NVDA', start='2024-01-01', end='2024-12-31', epochs=200):
    X, y, scaler, dates = get_timeseries(ticker, start, end)
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = PriceLSTM()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()

    for _ in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()

    return model, X_test, y_test, scaler, dates[split:]

# ----------------------------------------------------------------------
# Step 5: Evaluate and visualize predictions
# ----------------------------------------------------------------------
def evaluate_model(model, X_test, y_test, scaler, dates, ticker):
    model.eval()
    with torch.no_grad():
        preds = model(X_test)
    actual = scaler.inverse_transform(y_test)
    predicted = scaler.inverse_transform(preds.numpy())

    plt.figure(figsize=(12, 6))
    plt.plot(dates, actual, label="Actual", color="black")
    plt.plot(dates, predicted, label="Predicted", linestyle="--", color="gray")
    plt.title(f"{ticker} Next-Step Prediction on Historical Data")
    plt.xlabel("Date"); plt.ylabel("Price")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# ----------------------------------------------------------------------
# Step 6: Run the experiment
# ----------------------------------------------------------------------
model, X_test, y_test, scaler, dates = train_model(
    ticker="NVDA",
    start="2018-01-01",
    end="2023-12-31",
    epochs=50,
)
evaluate_model(model, X_test, y_test, scaler, dates, "NVDA")


#Listing 7-2: Comparing RNN, LSTM, and GRU in Practice
This code presents a compact, end-to-end sequence modeling demonstration comparing three recurrent architectures: a basic RNN, an LSTM, and a GRU. The code is intentionally small and self-contained so that each transformation is visible.

In [ ]:
#-----------------------------------------
# Step 1: Import dependencies
#-----------------------------------------
import re
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

#-----------------------------------------
# Step 2: Set random seeds for reproducibility
#-----------------------------------------
torch.manual_seed(42)
np.random.seed(42)

#-----------------------------------------
# Step 3: Define the training corpus
#-----------------------------------------
text = """
Mary had a little lamb
the cat sat on the mat
the dog sat on the log
the bird flew away
the fish swam in the pond
the quick brown fox jumped over the lazy dog
the cat climbed the tree
the dog chased the cat
the fox ran away
"""

#-----------------------------------------
# Step 4: Define a basic RNN model
#-----------------------------------------
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_size=16, hidden_size=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

#-----------------------------------------
# Step 5: Define an LSTM model
#-----------------------------------------
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size=16, hidden_size=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

#-----------------------------------------
# Step 6: Define a GRU model
#-----------------------------------------
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_size=16, hidden_size=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

#-----------------------------------------
# Step 7: Build the sequence prediction pipeline
#-----------------------------------------
class SequencePredictor:
    def __init__(self, text, seq_length=3):
        self.seq_length = seq_length
        self.words = self._preprocess(text)
        self.vocab = sorted(set(self.words))
        self.word2idx = {word: i for i, word in enumerate(self.vocab)}
        self.idx2word = {i: word for word, i in self.word2idx.items()}
        self.X_train, self.y_train = self._make_sequences()

    #-----------------------------------------
    # Step 8: Preprocess and normalize the text
    #-----------------------------------------
    def _preprocess(self, text):
        text = text.lower()
        text = re.sub(r"[^a-z\s]", "", text)
        return text.split()

    #-----------------------------------------
    # Step 9: Create supervised training sequences
    #-----------------------------------------
    def _make_sequences(self):
        X, y = [], []

        for i in range(len(self.words) - self.seq_length):
            context = self.words[i:i + self.seq_length]
            target = self.words[i + self.seq_length]

            X.append([self.word2idx[word] for word in context])
            y.append(self.word2idx[target])

        return torch.tensor(X), torch.tensor(y)

    #-----------------------------------------
    # Step 10: Train a recurrent model
    #-----------------------------------------
    def train(self, model, name, epochs=100, lr=0.01):
        print("-" * 70)
        print(f"Training {name} Model...")
        print("-" * 70)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(1, epochs + 1):
            optimizer.zero_grad()

            outputs = model(self.X_train)

            loss = criterion(outputs, self.y_train)

            loss.backward()

            optimizer.step()

            if epoch % 20 == 0:
                print(f"  Epoch {epoch}/100, Loss: {loss.item():.4f}")

        print()

    #-----------------------------------------
    # Step 11: Generate next-word predictions
    #-----------------------------------------
    def predict(self, model, phrase):
        model.eval()

        words = self._preprocess(phrase)

        while len(words) < self.seq_length:
            words.insert(0, "the")

        context = words[-self.seq_length:]

        indices = [
            self.word2idx.get(word, self.word2idx["the"])
            for word in context
        ]

        x = torch.tensor([indices])

        with torch.no_grad():
            logits = model(x)

            probs = torch.softmax(logits, dim=1)

            confidence, predicted_idx = torch.max(probs, dim=1)

        return self.idx2word[predicted_idx.item()], confidence.item()

#-----------------------------------------
# Step 12: Train and test all models
#-----------------------------------------
print("=" * 70)
print("PyTorch Sequence Modeling Demo: RNN vs LSTM vs GRU")
print("=" * 70)

print("\nInitializing models and preparing data...\n")

predictor = SequencePredictor(text, seq_length=3)

vocab_size = len(predictor.vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"Training sequences: {len(predictor.X_train)}")
print(f"Vocabulary: {predictor.vocab[:15]}...\n")

#-----------------------------------------
# Step 12a: Initialize the models
#-----------------------------------------
models = {
    "RNN": RNNModel(vocab_size),
    "LSTM": LSTMModel(vocab_size),
    "GRU": GRUModel(vocab_size),
}

#--------------------------------------------
# Step 12b: Train each recurrent architecture
#--------------------------------------------
for name, model in models.items():
    predictor.train(model, name)

print("=" * 70)
print("Training Complete! Now you can test the models.")
print("=" * 70)

#-----------------------------------------
# Step 12c: Run the interactive prediction loop
#-----------------------------------------
while True:
    print("-" * 70)

    phrase = input("\nEnter a phrase (or 'quit' to exit): ")

    if phrase.lower() == "quit":
        break

    print(f"\nInput phrase: '{phrase}'")

    print("\nPredictions:")

    predictions = {}

    for name, model in models.items():
        word, confidence = predictor.predict(model, phrase)

        predictions[name] = word

        print(f"  {name}:  '{word}' (confidence: {confidence:.2%})")

    print("\nComplete predictions:")

    for name, word in predictions.items():
        print(f"  {name}:  '{phrase} {word}'")

    print()

#-----------------------------------------
# Step 13: Exit the program
#-----------------------------------------
print("\nGoodbye!")

#Listing 7-3: A simple interactive example of sentiment analysis
This example uses a pretrained Transformer to classify sentiment. The code demonstrates inference, not training.

In [ ]:
# ---------------------------------------------
# Step 1. Setup and Imports
# ---------------------------------------------
!pip install transformers --quiet

from transformers import pipeline

# ---------------------------------------------
# Step 2. Load a sentiment-analysis pipeline
#         (a default sentiment model is loaded if none is specified)
# ---------------------------------------------
classifier = pipeline("sentiment-analysis")

# ---------------------------------------------
# Step 3. Print User Instructions
# ---------------------------------------------
print("Enter a message to analyze its sentiment.")
print("Type 'quit' to exit.\n")

# ---------------------------------------------
# Step 4. Interactive Input Loop
# ---------------------------------------------
while True:

    # ---------------------------------------------
    # Step 4a. Read and normalize user input
    # ---------------------------------------------
    s = input("Your message: ").strip()

    # ---------------------------------------------
    # Step 4b. Exit condition
    # ---------------------------------------------
    if s.lower() in {"quit", "exit"}:
        break

    # ---------------------------------------------
    # Step 4c. Basic validation: reject empty input
    # ---------------------------------------------
    if not s:
        print("Please type a non-empty message.\n")
        continue

    # ---------------------------------------------
    # Step 5. Run Transformer Inference
    # ---------------------------------------------
    result = classifier(s)[0]

    # ---------------------------------------------
    # Step 6. Extract and Present the Result
    # ---------------------------------------------
    label = result["label"]
    score = result["score"] * 100
    print(f"→ {label} ({score:.1f}%)\n")


#Listing 7-4: Fine-tuning a pretrained Transformer for 3-class customer feedback
This example demonstrates how a pretrained Transformer can be adapted to a new classification task through fine-tuning. The model architecture itself does not change; instead, we redefine the prediction task and adjust the model’s parameters so its learned representations align with a new label space and dataset.

In [ ]:
# -----------------------------------------------------------
# Step 1. Setup and Imports
# -----------------------------------------------------------
!pip install transformers datasets evaluate accelerate --quiet

import numpy as np
import torch
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------
# Step 2. Define Labels (the label space is a design choice)
# ----------------------------------------------------------
id2label = {0: "POSITIVE", 1: "MILD_CRITICISM", 2: "SEVERE_COMPLAINT"}
label2id = {v: k for k, v in id2label.items()}

# -----------------------------------------------------------
# Step 3. Create a Small Labeled Dataset (toy example)
# -----------------------------------------------------------
train_examples = [
    {"text": "Love the screen and the speed. No complaints.", "label": label2id["POSITIVE"]},
    {"text": "Fast and beautiful, but the battery barely lasts a day.", "label": label2id["MILD_CRITICISM"]},
    {"text": "Stopped working after two days. Support was useless.", "label": label2id["SEVERE_COMPLAINT"]},
    {"text": "Great camera and smooth performance.", "label": label2id["POSITIVE"]},
    {"text": "Good phone overall, but the battery could be better.", "label": label2id["MILD_CRITICISM"]},
    {"text": "Terrible build quality. Broke within a week.", "label": label2id["SEVERE_COMPLAINT"]},
]

eval_examples = [
    {"text": "Excellent performance and a great display.", "label": label2id["POSITIVE"]},
    {"text": "Nice design, but the battery drains quickly.", "label": label2id["MILD_CRITICISM"]},
    {"text": "Completely unreliable. Would not recommend.", "label": label2id["SEVERE_COMPLAINT"]},
]

train_ds = Dataset.from_list(train_examples)
eval_ds = Dataset.from_list(eval_examples)

# -----------------------------------------------------------
# Step 4. Tokenize (sequence definition is a design choice)
# -----------------------------------------------------------
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

MAX_LEN = 128  # Adjust based on typical review length and available compute.

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True)
eval_ds = eval_ds.map(tokenize, batched=True)

# Keep only columns used by the model + Trainer
train_ds = train_ds.remove_columns(["text"])
eval_ds = eval_ds.remove_columns(["text"])

# Tell Datasets to return PyTorch tensors for these columns
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------------------------------------------
# Step 5. Load a Pretrained Model and Configure the Head
# -----------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(id2label),          # 3
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,      # allow 2 -> 3 head swap
).to(device)

# -----------------------------------------------------------
# Step 6. Train and Evaluate
# -----------------------------------------------------------
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir="sentiment_ft",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

trainer.train()
trainer.evaluate()

# -----------------------------------------------------------
# Step 7. Run Inference with the Fine-Tuned Model
# -----------------------------------------------------------
def predict(text: str) -> str:
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred_id = int(outputs.logits.argmax(dim=-1).item())
    return id2label[pred_id]

# -----------------------------------------------------------
# Step 8. Display output
# -----------------------------------------------------------
print(predict("The phone is fast, but the battery barely lasts a day."))
print(predict("Stopped working after two days. Support was useless."))
print(predict("Great camera, great performance."))


#Listing 7-5: A minimal interactive question-answering example using a pretrained Transformer
This listing shows a simple question-answering pipeline. The model receives a context passage and a question, then returns the most relevant answer span.

In [ ]:
# ---------------------------------------------
# Step 1. Import the pipeline helper
# ---------------------------------------------
from transformers import pipeline

# ---------------------------------------------
# Step 2. Load a question-answering pipeline
#         (default pretrained model)
# ---------------------------------------------
qa = pipeline("question-answering")

# ---------------------------------------------
# Step 3. Collect user input
# ---------------------------------------------
context = input("Enter context: ")
question = input("Now enter your question: ")

# ---------------------------------------------
# Step 4. Run Transformer inference
# ---------------------------------------------
result = qa(question=question, context=context)

# ---------------------------------------------
# Step 5. Extract and display the answer
# ---------------------------------------------
print(f"Answer: {result['answer']} (score: {result['score']:.2f})")

#Listing 7-6: A compact implementation of the chat-with-my-data pattern
This example uses a Transformer-based question-answering pipeline from Hugging Face and a lightweight PDF reader (PyMuPDF).


In [ ]:
# ---------------------------------------------
# Step 1. Install required libraries (Colab)
# ---------------------------------------------
# transformers: pretrained Transformer models + pipelines
# PyMuPDF (fitz): PDF parsing and text extraction
!pip install transformers --quiet
!pip install PyMuPDF --quiet

# ---------------------------------------------
# Step 2. Import dependencies
# ---------------------------------------------
from transformers import pipeline
import fitz  # PyMuPDF
from google.colab import files

# ---------------------------------------------
# Step 3. Upload a PDF from your local machine
# ---------------------------------------------
# Colab returns a dict: {filename: bytes}
uploaded = files.upload()

# Take the first uploaded file
file_name = next(iter(uploaded))
pdf_data = uploaded[file_name]

# ---------------------------------------------
# Step 4. Extract text from the uploaded PDF
# ---------------------------------------------
def extract_text_from_pdf_bytes(pdf_bytes):
    """Extract all page text from a PDF stored as raw bytes."""
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

context = extract_text_from_pdf_bytes(pdf_data)

# ---------------------------------------------
# Step 5. Initialize a question-answering pipeline
# ---------------------------------------------
qa = pipeline("question-answering")

# ---------------------------------------------
# Step 6. Interactive Q&A loop over the PDF text
# ---------------------------------------------
print("\nPDF loaded. Ask me anything about its contents. Type 'quit' to exit.")

while True:
    # -----------------------------------------
    # Step 6a. Read and validate the question
    # -----------------------------------------
    question = input("\nYour question: ").strip()

    if question.lower() == "quit":
        print("Exiting Q&A.")
        break

    if len(question) == 0:
        print("Please enter a valid question.")
        continue

    # -----------------------------------------
    # Step 6b. Run extractive QA inference
    # -----------------------------------------
    try:
        result = qa(question=question, context=context)

        # -------------------------------------
        # Step 6c. Present the answer span
        # -------------------------------------
        print(f"Answer: {result['answer']} (score: {result['score']:.2f})")

    except Exception as e:
        # -------------------------------------
        # Step 6d. Basic error handling
        # -------------------------------------
        print(f"Could not answer the question. Reason: {e}")
